In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


In [ ]:
import os

# Caminho para salvar o gráfico e o arquivo CSV
save_path = '../../results/regression/graphics'
os.makedirs(save_path, exist_ok=True)  # Cria o diretório, caso não exista

# Filtragem e visualização
bbr = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
columns_to_drop = ['PolynomialRegression', 'AdaBoostRegressor', 'ElasticNet', 
                   'LinearRegression', 'MLPRegressor', 'SVR', 'KNeighborsRegressor']

sources_to_drop = ['df']#, 'rj', 'sp', 'pa', 'sc', 'pr', 'mg'] #lembrar de semre excluir o df, pois o dataset ficou pequeno 

bbr = bbr.drop(columns=columns_to_drop, errors='ignore')
bbr = bbr[~bbr['source'].isin(sources_to_drop)]

models = bbr.columns[1:]
y = np.arange(len(bbr['source']))  # Agora as categorias estão no eixo y
height = 0.7 / len(models)         # Ajustando altura das barras para maior largura - 0.7

custom_colors = [
    'blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'maroon', 'brown'
    #'olive', 'peru', 'gold', 'deeppink', 'lime', 'royalblue', 'darkviolet'
]

# Ajustando o tamanho da figura para ser menor em largura e maior em comprimento
fig, ax = plt.subplots(figsize=(10, 12))  

for i, model in enumerate(models):
    # Usando as cores definidas manualmente
    ax.barh(y + i * height, bbr[model], height, label=model, color=custom_colors[i], edgecolor='black')  # Adicionando borda preta nas barras

ax.set_title("Comparação de RMSE por Modelo e Ponto de Comunicação", fontsize=14)
ax.set_xlabel("NRMSE", fontsize=12)  # Eixo x agora representa os valores
ax.set_ylabel("Ponto de Comunicação", fontsize=12)  # Eixo y representa as categorias
ax.set_yticks(y + height * (len(models) / 2 - 0.5))
ax.set_yticklabels(bbr['source'].str.upper())

# Ajustando o espaço abaixo do gráfico para a legenda
plt.subplots_adjust(bottom=0.15)

# Legenda ajustada para ficar mais próxima do gráfico
ax.legend(
    title="Modelos de Regressão",
    bbox_to_anchor=(0.5, -0.05),  # Ajusta a posição para ficar mais próxima
    loc='upper center',
    ncol=3
)

ax.grid(axis='x', linestyle='--', alpha=0.7)  # Grid no eixo x
plt.tight_layout()

# Salvar o gráfico
#graph_path = os.path.join(save_path, '5-modelos-todos-links-ruins.png')
#plt.savefig(graph_path, dpi=300)
#print(f"Gráfico salvo em: {graph_path}")

plt.show()



# Métricas estatísticas:
#                             nrmse                                
#                              mean     std  median     min     max
# model                                                            
# AdaBoostRegressor          0.2676  0.3231  0.0591  0.0359  0.9362
# CatBoostRegressor          0.2224  0.2615  0.0511  0.0314  0.7054
# ElasticNet                 0.2934  0.3607  0.0609  0.0379  1.0186
# GradientBoostingRegressor  0.2310  0.2735  0.0512  0.0315  0.7379
# KNeighborsRegressor        0.2470  0.2870  0.0570  0.0358  0.7750
# LGBMRegressor              0.2241  0.2627  0.0513  0.0318  0.7195
# LinearRegression           0.2934  0.3607  0.0609  0.0379  1.0186
# MLPRegressor               0.3439  0.3611  0.0924  0.0379  1.0381
# PolynomialRegression       0.2711  0.3258  0.0560  0.0360  0.9310
# RandomForestRegressor      0.2275  0.2671  0.0518  0.0317  0.7224
# SVR                        0.3548  0.4437  0.0657  0.0399  1.2578
# XGBRegressor               0.2285  0.2701  0.0520  0.0316  0.7304

In [ ]:
df1 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_bbr.csv')
df2 = pd.read_csv('../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_Vazao_cubic.csv')

In [ ]:
# quero percorrer uma pasta que contem varios csv (rferentes a minha source), 
# cada arquivo tem uma coluna de y_test e as outras colunas sao os valores de y_pred para cada modelo
# quero categorizar cada y_test e y_pred (modelo por modelo) e verificar se eles estao na mesma categoria e salva isso em porcentagem 
# de acertos. quetro que faca isso recursivamente para todos os meus arquivose links. 
#apos isso quero plotar um grafico de barras em que as barras estarao juntas por source (cada source vai ter barras comparando o modelo)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# def categorize_nrmse(value):
#     if value < 0.1:
#         return 'Excelente'
#     elif value < 0.2:
#         return 'Bom'
#     elif value < 0.3:
#         return 'Razoável'
#     elif value < 0.5:
#         return 'Ruim'
#     else:
#         return 'Muito Ruim'
def categorize_nrmse(value):
    if value < 0:
        return 'Valor Inválido'
    elif value < 0.1:
        return 'Excelente'
    elif value < 0.2:
        return 'Bom'
    elif value < 0.3:
        return 'Razoável'
    elif value < 0.5:
        return 'Ruim'
    else:
        return 'Muito Ruim'


def analyze_nrmse(df):
    df_melted = df.melt(id_vars=['source'], var_name='model', value_name='nrmse')
    
    # Adicionar categorias
    df_melted['categoria'] = df_melted['nrmse'].apply(categorize_nrmse)
    
    # 1. Análise geral por modelo
    model_analysis = df_melted.groupby('model').agg({
        'nrmse': ['mean', 'std', 'median', 'min', 'max']
    }).round(4)
    
    # 2. Distribuição das categorias por modelo
    category_dist = pd.crosstab(df_melted['model'], df_melted['categoria'], normalize='index') * 100
    
    # 3. Identificar melhores estados por modelo
    best_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmin()]
    
    # 4. Identificar piores estados por modelo
    worst_states = df_melted.loc[df_melted.groupby('model')['nrmse'].idxmax()]
    
    return df_melted, model_analysis, category_dist, best_states, worst_states

def plot_analysis(df_melted, model_analysis, category_dist):
    """
    Cria visualizações para a análise do NRMSE.
    """
    plt.style.use('default')
    
    fig = plt.figure(figsize=(20, 15))
    
    # # 1. Boxplot dos modelos
    # plt.subplot(2, 2, 1)
    # sns.boxplot(data=df_melted, x='model', y='nrmse', width=0.7)
    # plt.xticks(rotation=45, ha='right')
    # plt.title('Distribuição do NRMSE por Modelo')
    # plt.xlabel('Modelo')
    # plt.ylabel('NRMSE')
    
    # 2. Heatmap da distribuição de categorias
    plt.subplot(2, 2, 2)
    sns.heatmap(category_dist, annot=True, fmt='.1f', cmap='YlOrRd')
    plt.title('Distribuição das Categorias por Modelo (%)')
    plt.xlabel('Categoria')
    plt.ylabel('Modelo')
    
    # # 3. Gráfico de barras do NRMSE médio
    # plt.subplot(2, 2, 3)
    # model_means = model_analysis['nrmse']['mean'].sort_values()
    # plt.bar(range(len(model_means)), model_means)
    # plt.xticks(range(len(model_means)), model_means.index, rotation=45, ha='right')
    # plt.title('NRMSE Médio por Modelo')
    # plt.xlabel('Modelo')
    # plt.ylabel('NRMSE Médio')
    
    # 4. Gráfico de dispersão do NRMSE por estado
    plt.subplot(2, 2, 4)
    markers = ['o', 's', '^', 'v', 'D', 'p', 'h', '8', '*', '+', 'x', 'd']
    for i, model in enumerate(df_melted['model'].unique()):
        model_data = df_melted[df_melted['model'] == model]
        plt.scatter(model_data['source'], model_data['nrmse'], 
                   label=model, alpha=0.6, marker=markers[i % len(markers)])
    plt.xticks(rotation=45, ha='right')
    plt.title('NRMSE por Estado e Modelo')
    plt.xlabel('Estado')
    plt.ylabel('NRMSE')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    return fig


df = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')


df_melted, model_analysis, category_dist, best_states, worst_states = analyze_nrmse(df)

fig = plot_analysis(df_melted, model_analysis, category_dist)

# Imprimir resultados detalhados
print("\n=== Análise por Modelo ===")
print("\nMétricas estatísticas:")
print(model_analysis)

print("\n=== Melhores Estados por Modelo ===")
best_states_formatted = best_states[['model', 'source', 'nrmse']].sort_values('nrmse')
print(best_states_formatted.to_string())

print("\n=== Piores Estados por Modelo ===")
worst_states_formatted = worst_states[['model', 'source', 'nrmse']].sort_values('nrmse', ascending=False)
print(worst_states_formatted.to_string())

# Calcular e mostrar distribuição geral das categorias
total_dist = df_melted['categoria'].value_counts(normalize=True) * 100
print("\n=== Distribuição Geral das Categorias ===")
print(total_dist.round(2).sort_index())

# Salvar resultados em arquivos
output_dir = '../../results/regression/analysis'
os.makedirs(output_dir, exist_ok=True)

model_analysis.to_csv(f'{output_dir}/model_analysis_bbr.csv')
category_dist.to_csv(f'{output_dir}/category_distribution_bbr.csv')
plt.savefig(f'{output_dir}/nrmse_analysis_plots_bbr.png', bbox_inches='tight', dpi=300)

# Salvar também um resumo em formato mais legível
with open(f'{output_dir}/analysis_summary_bbr.txt', 'w') as f:
    f.write("=== Análise de NRMSE ===\n\n")
    f.write("Distribuição das Categorias:\n")
    f.write(total_dist.round(2).sort_index().to_string())
    f.write("\n\nMelhores Estados:\n")
    f.write(best_states_formatted.to_string())
    f.write("\n\nPiores Estados:\n")
    f.write(worst_states_formatted.to_string())

In [ ]:
# Coeficiente de variância dos valores de vazao 
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df
dataframes_by_source.pop('df', None) #df tem poucas linhas, entao a gente resolveu descatra isso 

protocol = 'Vazao_bbr'
rmse = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse = rmse[~(rmse['source'] == 'df')]

correlations_by_model = {}
variances = {}

for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
    variance_coef = (dataset[protocol].std() / dataset[protocol].mean()) * 100
    variances[key] = variance_coef

models = rmse.columns[1:]

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])  
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

# Exibindo as correlações
print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model}: {correlation:.4f}")




# Convertendo o dicionário em DataFrame
correlations_df = pd.DataFrame(list(correlations_by_model.items()), 
                             columns=['Model', 'Correlation'])
correlations_df = correlations_df.sort_values('Correlation', ascending=True)

# Configurando o tema do seaborn
sns.set_theme(style="whitegrid")

# Criando os gráficos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Gráfico de barras horizontal usando barplot do seaborn
sns.barplot(y='Model', x='Correlation', data=correlations_df, ax=ax1,
           palette='viridis', orient='h')
ax1.set_title('Correlation between RMSE and Dataset Variance by Model')
ax1.set_xlabel('Correlation Coefficient')

# 2. Heatmap das correlações
heatmap_data = correlations_df.set_index('Model')
sns.heatmap(heatmap_data.T, annot=True, cmap='RdYlBu', center=0, 
            fmt='.3f', ax=ax2)
ax2.set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

# Scatter plots para cada modelo
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, model in enumerate(correlations_by_model.keys()):
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])
    })
    
    sns.scatterplot(data=df_model, x='Variance_coef', y='RMSE', ax=axes[idx])
    axes[idx].set_title(f'{model}\nCorr: {correlations_by_model[model]:.3f}')
    axes[idx].set_xlabel('Variance Coefficient (%)')
    axes[idx].set_ylabel('RMSE')

plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(12, 6))
rmse_melted = rmse.melt(id_vars=['source'], 
                        var_name='Model', 
                        value_name='RMSE')
sns.boxplot(x='Model', y='RMSE', data=rmse_melted, palette='viridis')
plt.xticks(rotation=45)
plt.title('Distribution of RMSE by Model')
plt.tight_layout()
plt.show()

In [ ]:
#testar as predições em arquivos separados - related

import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

def normalized_rmse(y_actual, y_predict):
    """Calculate normalized RMSE."""
    rmse = np.sqrt(mean_squared_error(y_actual, y_predict))
    mean_actual = np.mean(y_actual)
    return rmse / mean_actual

def process_predictions_folder(input_folder, target="bbr", output_file="normalized_rmse_results.csv"):
    """Process all prediction files in the folder and calculate NRMSE for each model."""
    results = []
    
    # Percorre os arquivos na pasta
    for filename in os.listdir(input_folder):
        # Verifica se é um arquivo related e contém o target especificado
        if "predictions" in filename and target in filename and filename.endswith(".csv"):
            file_path = os.path.join(input_folder, filename)
            
            # Lê o arquivo CSV
            df = pd.read_csv(file_path)
            
            # Identifica a coluna de valores reais
            y_actual_col = "y_actual"
            
            # Lista todas as colunas de predição (para todos os modelos)
            predict_cols = [col for col in df.columns if col.startswith("y_predict_")]
            
            if y_actual_col in df.columns and predict_cols:
                y_actual = df[y_actual_col]
                
                # Para cada modelo, calcula o NRMSE
                for predict_col in predict_cols:
                    # Extrai o nome do modelo da coluna
                    model_name = predict_col.replace("y_predict_", "")
                    
                    # Calcula o NRMSE
                    y_predict = df[predict_col]
                    norm_rmse = normalized_rmse(y_actual, y_predict)
                    
                    # Extrai informações do nome do arquivo
                    file_parts = filename.split("_")
                    scenario = file_parts[2] if len(file_parts) > 2 else "unknown"
                    dataset = file_parts[-1].replace(".csv", "") if len(file_parts) > 3 else "unknown"
                    
                    # Adiciona o resultado à lista
                    results.append({
                        "filename": filename,
                        "scenario": scenario,
                        "dataset": dataset,
                        "model": model_name,
                        "normalized_rmse": norm_rmse
                    })
    
    # Cria DataFrame com os resultados
    results_df = pd.DataFrame(results)
    
    # Pivota o DataFrame para ter os modelos como colunas
    pivot_df = results_df.pivot_table(
        index=['filename', 'scenario', 'dataset'],
        columns='model',
        values='normalized_rmse'
    ).reset_index()
    
    # Ordena as colunas para melhor visualização
    cols_order = ['filename', 'scenario', 'dataset'] + sorted([col for col in pivot_df.columns if col not in ['filename', 'scenario', 'dataset']])
    pivot_df = pivot_df[cols_order]
    
    # Salva os resultados
    pivot_df.to_csv(output_file, index=False)
    print(f"Resultados salvos em {output_file}")
    
    return pivot_df

# Executa o processamento
input_folder = '../../results/regression/predictions-related'
output_file = "normalized_rmse_results_bbr.csv"
results_df = process_predictions_folder(input_folder, target="bbr", output_file=output_file)

# Mostra as primeiras linhas do resultado
print("\nPrimeiras linhas dos resultados:")
print(results_df.head())

# Mostra estatísticas resumidas por modelo
print("\nEstatísticas resumidas por modelo:")
model_stats = results_df.select_dtypes(include=[np.number]).describe()
print(model_stats)

In [ ]:
model_columns = [
    "CatBoostRegressor",
    "GradientBoostingRegressor",
    "LGBMRegressor",
    "RandomForestRegressor",
    "XGBRegressor"
]
filtered_df010 = results_df[results_df[model_columns].lt(0.10).any(axis=1)]
# Verifica se pelo menos uma dessas colunas está entre 0.10 e 0.20
filtered_df020 = results_df[results_df[model_columns].apply(
    lambda x: (x > 0.10) & (x < 0.20)
).any(axis=1)]

# Mostra o resultado
# print("Linhas com pelo menos uma coluna entre 0.10 e 0.20:")
# print(len(filtered_df020))
# Mostra o resultado
print("Linhas com pelo menos uma coluna menor que 0.10:")
print(filtered_df010)

# # # Salva o resultado em um arquivo CSV, se necessário
# filtered_df010.to_csv("filtered_results_below_0.10.csv", index=False)
# print("Resultados salvos em 'filtered_results_below_0.10.csv'")

In [ ]:
#aqui eu avalio a predicao de cada dataset original e adiciono a coluna de coeficiente de varianvcia e tamanho
#do dataset
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import re  # Para verificar o padrão da substring

def normalized_rmse(y_actual, y_predict):
    """Calculate normalized RMSE."""
    rmse = np.sqrt(mean_squared_error(y_actual, y_predict))
    mean_actual = np.mean(y_actual)
    return rmse / mean_actual

def calculate_variation_coefficient(series):
    """Calculate coefficient of variation for a series."""
    mean = np.mean(series)
    std = np.std(series)
    # Retorna NaN se a média for zero, caso contrário retorna o CV
    return std / mean if mean != 0 else np.nan

def process_predictions_folder(input_folder, original_data_folder, target_substring="bbr", output_file="normalized_rmse_results.csv"):
    """Process prediction files and include the coefficient of variation and dataset size from the original datasets."""
    results = []
    
    # Define o padrão de duas letras seguidas por hífen e outras duas letras
    pattern = r"-[a-z]{2}-[a-z]{2}"
    
    # Percorre os arquivos na pasta de predições
    for filename in os.listdir(input_folder):
        if "predictions" in filename and filename.endswith(".csv"):
            file_path = os.path.join(input_folder, filename)
            
            # Lê o arquivo de predições
            pred_df = pd.read_csv(file_path)
            
            # Identifica a coluna de valores reais
            y_actual_col = "y_actual"
            predict_cols = [col for col in pred_df.columns if col.startswith("y_predict_")]
            
            if y_actual_col in pred_df.columns and predict_cols:
                y_actual = pred_df[y_actual_col]
                
                # Extrai a substring no formato esperado
                match = re.search(pattern, filename)
                common_substring = match.group() if match else None
                
                if not common_substring:
                    print(f"Não foi possível identificar a substring no formato esperado em {filename}. Pulando...")
                    continue
                
                # Localiza o arquivo original correspondente
                original_file = next((f for f in os.listdir(original_data_folder) if common_substring in f), None)
                
                if original_file:
                    original_path = os.path.join(original_data_folder, original_file)
                    original_df = pd.read_csv(original_path)
                    
                    # Localiza a coluna que contém a substring 'bbr'
                    target_column = next((col for col in original_df.columns if target_substring in col), None)
                    
                    if target_column:
                        # Calcula o coeficiente de variação da coluna com a substring 'bbr'
                        variation_coefficient = calculate_variation_coefficient(original_df[target_column])
                    else:
                        variation_coefficient = np.nan
                    
                    # Determina o tamanho do dataset original
                    dataset_size = len(original_df)
                else:
                    variation_coefficient = np.nan
                    dataset_size = np.nan
                
                # Para cada modelo, calcula o NRMSE
                for predict_col in predict_cols:
                    model_name = predict_col.replace("y_predict_", "")
                    y_predict = pred_df[predict_col]
                    norm_rmse = normalized_rmse(y_actual, y_predict)
                    
                    # Adiciona os resultados à lista
                    results.append({
                        "filename": filename,
                        "common_substring": common_substring,
                        "model": model_name,
                        "normalized_rmse": norm_rmse,
                        "variation_coefficient": variation_coefficient,  # Agora inclui o coeficiente de variação
                        "dataset_size": dataset_size  # Mantém o dataset_size
                    })

    # Cria DataFrame com os resultados
    results_df = pd.DataFrame(results)
    
    # Pivota o DataFrame para os modelos como colunas
    pivot_df = results_df.pivot_table(
        index=['filename', 'common_substring'],
        columns='model',
        values=['normalized_rmse']
    ).reset_index()

    # "Flatten" os múltiplos níveis de colunas
    pivot_df.columns = [f'{col[0]}_{col[1]}' if col[1] else col[0] for col in pivot_df.columns]

    # Adiciona as colunas 'dataset_size' e 'variation_coefficient' para cada combinação de 'filename' e 'common_substring'
    size_and_variation_df = results_df[['filename', 'common_substring', 'dataset_size', 'variation_coefficient']].drop_duplicates()

    # Merge para adicionar dataset_size e variation_coefficient ao pivot_df
    final_df = pd.merge(pivot_df, size_and_variation_df, on=['filename', 'common_substring'], how='left')

    # Organiza as colunas
    final_df.sort_values(by="filename", inplace=True)

    # Salva os resultados
    final_df.to_csv(output_file, index=False)
    print(f"Resultados salvos em {output_file}")
    
    return final_df

# Caminhos das pastas
input_folder = '../../results/regression/teste-predictions-related'
original_data_folder = '../../datasets/serie-multivariada'
output_file = "normalized_rmse_with_variation_coefficient_and_size.csv"

# Executa o processamento
results_df = process_predictions_folder(input_folder, original_data_folder, target_substring="bbr", output_file=output_file)


In [ ]:
results_df

In [ ]:
#agr preciso filtrar os pontos de comunicacao que tiveram bons rmse no geral, e filtrar esse aqui com base nos bons resultados 
#como se fosse a aplicacao da ferramenta em dataset que foram bons 
# plotar o grafico de dispersao 

In [ ]:
#correlacao entre o nrmse e o coeficiente de variancia do dataset original 

# Suponha que o seu DataFrame seja chamado df
df = results_df

# Calcular a correlação entre o coeficiente de variação e o NRMSE para cada modelo
correlation_variation_nrmse = {
    'CatBoostRegressor': df['variation_coefficient'].corr(df['normalized_rmse_CatBoostRegressor']),
    'GradientBoostingRegressor': df['variation_coefficient'].corr(df['normalized_rmse_GradientBoostingRegressor']),
    'LGBMRegressor': df['variation_coefficient'].corr(df['normalized_rmse_LGBMRegressor']),
    'RandomForestRegressor': df['variation_coefficient'].corr(df['normalized_rmse_RandomForestRegressor']),
    'XGBRegressor': df['variation_coefficient'].corr(df['normalized_rmse_XGBRegressor'])
}

# Calcular a correlação entre o tamanho do dataset e o NRMSE para cada modelo
correlation_dataset_size_nrmse = {
    'CatBoostRegressor': df['dataset_size'].corr(df['normalized_rmse_CatBoostRegressor']),
    'GradientBoostingRegressor': df['dataset_size'].corr(df['normalized_rmse_GradientBoostingRegressor']),
    'LGBMRegressor': df['dataset_size'].corr(df['normalized_rmse_LGBMRegressor']),
    'RandomForestRegressor': df['dataset_size'].corr(df['normalized_rmse_RandomForestRegressor']),
    'XGBRegressor': df['dataset_size'].corr(df['normalized_rmse_XGBRegressor'])
}

# Exibir os resultados
print("Correlação entre o coeficiente de variação e o NRMSE para cada modelo:")
print(correlation_variation_nrmse)

print("\nCorrelação entre o tamanho do dataset e o NRMSE para cada modelo:")
print(correlation_dataset_size_nrmse)


In [ ]:
import os
#codigo original, mas embaralhado
# Caminho para salvar o gráfico e o arquivo CSV
save_path = '../../results/regression/graphics'
os.makedirs(save_path, exist_ok=True)  # Cria o diretório, caso não exista

# Filtragem e visualização
bbr = pd.read_csv('../../results/regression/teste-predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
columns_to_drop = ['PolynomialRegression', 'AdaBoostRegressor', 'ElasticNet', 
                   'LinearRegression', 'MLPRegressor', 'SVR', 'KNeighborsRegressor']

sources_to_drop = ['df']#, 'rj', 'sp', 'pa', 'sc', 'pr', 'mg'] #lembrar de semre excluir o df, pois o dataset ficou pequeno 

bbr = bbr.drop(columns=columns_to_drop, errors='ignore')
#bbr = bbr[~bbr['source'].isin(sources_to_drop)]

models = bbr.columns[1:]
y = np.arange(len(bbr['source']))  # Agora as categorias estão no eixo y
height = 0.7 / len(models)         # Ajustando altura das barras para maior largura - 0.7

custom_colors = [
    'blue', 'green', 'red', 'purple', 'orange', 'cyan', 'magenta', 'yellow', 'maroon', 'brown'
    #'olive', 'peru', 'gold', 'deeppink', 'lime', 'royalblue', 'darkviolet'
]

# Ajustando o tamanho da figura para ser menor em largura e maior em comprimento
fig, ax = plt.subplots(figsize=(10, 12))  

for i, model in enumerate(models):
    # Usando as cores definidas manualmente
    ax.barh(y + i * height, bbr[model], height, label=model, color=custom_colors[i], edgecolor='black')  # Adicionando borda preta nas barras

ax.set_title("Comparação de RMSE por Modelo e Ponto de Comunicação", fontsize=14)
ax.set_xlabel("NRMSE", fontsize=12)  # Eixo x agora representa os valores
ax.set_ylabel("Ponto de Comunicação", fontsize=12)  # Eixo y representa as categorias
ax.set_yticks(y + height * (len(models) / 2 - 0.5))
ax.set_yticklabels(bbr['source'].str.upper())

# Ajustando o espaço abaixo do gráfico para a legenda
plt.subplots_adjust(bottom=0.15)

# Legenda ajustada para ficar mais próxima do gráfico
ax.legend(
    title="Modelos de Regressão",
    bbox_to_anchor=(0.5, -0.05),  # Ajusta a posição para ficar mais próxima
    loc='upper center',
    ncol=3
)

ax.grid(axis='x', linestyle='--', alpha=0.7)  # Grid no eixo x
plt.tight_layout()


plt.show()
